# Knowledge Graph Construction and Reasoning

Vector databases retrieve documents based on semantic similarity, but they struggle with structured relationships and multi-hop reasoning. Knowledge graphs store information as entities and relationships, enabling queries like "Who founded the company that acquired Instagram?"

In this notebook, we'll:
- Extract entities and relationships from text using LLMs
- Build a knowledge graph with NetworkX
- Query the graph with natural language
- Compare graph-based QA with pure RAG

In [ ]:
%pip install -q openai pydantic networkx

In [ ]:
import os
import json
from getpass import getpass
from dataclasses import dataclass
from openai import OpenAI
from pydantic import BaseModel, Field
import networkx as nx

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-5-mini"

## Part 1: Entity Extraction

First, we extract named entities from text. Entities are the "nouns" of our knowledge graph.

In [ ]:
class Entity(BaseModel):
    name: str = Field(description="The entity name")
    type: str = Field(description="Entity type: PERSON, ORGANIZATION, LOCATION, PRODUCT, EVENT, DATE, OTHER")

class EntityList(BaseModel):
    entities: list[Entity] = Field(description="List of extracted entities")

def extract_entities(text: str) -> list[Entity]:
    """Extract named entities from text."""
    
    schema = EntityList.model_json_schema()
    schema["additionalProperties"] = False
    if "$defs" in schema:
        for def_schema in schema["$defs"].values():
            def_schema["additionalProperties"] = False
    
    response = client.responses.create(
        model=MODEL,
        input=f"""Extract all named entities from this text. Include people, organizations, locations, products, events, and dates.

Text: {text}

Extract entities:""",
        text={
            "format": {
                "type": "json_schema",
                "name": "EntityList",
                "strict": True,
                "schema": schema
            }
        }
    )
    
    result = EntityList.model_validate_json(response.output_text)
    return result.entities

# Test
sample_text = """
Mark Zuckerberg founded Facebook in 2004 at Harvard University. 
In 2012, Facebook acquired Instagram for $1 billion. 
The company later rebranded to Meta in 2021.
"""

entities = extract_entities(sample_text)
print("Extracted Entities:")
for e in entities:
    print(f"  {e.name} ({e.type})")

## Part 2: Relationship Extraction

Now we extract relationships between entities. Relationships are triples: (subject, predicate, object).

In [ ]:
class Relationship(BaseModel):
    subject: str = Field(description="The subject entity")
    predicate: str = Field(description="The relationship type (e.g., FOUNDED, ACQUIRED, LOCATED_IN)")
    object: str = Field(description="The object entity")

class RelationshipList(BaseModel):
    relationships: list[Relationship] = Field(description="List of extracted relationships")

def extract_relationships(text: str) -> list[Relationship]:
    """Extract relationships between entities."""
    
    schema = RelationshipList.model_json_schema()
    schema["additionalProperties"] = False
    if "$defs" in schema:
        for def_schema in schema["$defs"].values():
            def_schema["additionalProperties"] = False
    
    response = client.responses.create(
        model=MODEL,
        input=f"""Extract all relationships between entities in this text.
Each relationship should be a triple: (subject, predicate, object)

Use standardized predicates like: FOUNDED, ACQUIRED, WORKS_AT, LOCATED_IN, CREATED, RENAMED_TO, HAPPENED_ON, etc.

Text: {text}

Extract relationships:""",
        text={
            "format": {
                "type": "json_schema",
                "name": "RelationshipList",
                "strict": True,
                "schema": schema
            }
        }
    )
    
    result = RelationshipList.model_validate_json(response.output_text)
    return result.relationships

# Test
relationships = extract_relationships(sample_text)
print("\nExtracted Relationships:")
for r in relationships:
    print(f"  ({r.subject}) --[{r.predicate}]--> ({r.object})")

## Part 3: Building the Knowledge Graph

We'll use NetworkX to create and manage the graph structure.

In [ ]:
@dataclass
class KnowledgeGraph:
    """Simple knowledge graph using NetworkX."""
    
    def __post_init__(self):
        self.graph = nx.DiGraph()  # Directed graph for relationships
        self.entity_types = {}  # entity_name -> type
    
    def add_entity(self, name: str, entity_type: str):
        """Add an entity node."""
        self.graph.add_node(name)
        self.entity_types[name] = entity_type
    
    def add_relationship(self, subject: str, predicate: str, obj: str):
        """Add a relationship edge."""
        # Ensure nodes exist
        if subject not in self.graph:
            self.graph.add_node(subject)
        if obj not in self.graph:
            self.graph.add_node(obj)
        
        self.graph.add_edge(subject, obj, predicate=predicate)
    
    def from_text(self, text: str):
        """Build graph from text using LLM extraction."""
        # Extract and add entities
        entities = extract_entities(text)
        for e in entities:
            self.add_entity(e.name, e.type)
        
        # Extract and add relationships
        relationships = extract_relationships(text)
        for r in relationships:
            self.add_relationship(r.subject, r.predicate, r.object)
        
        return len(entities), len(relationships)
    
    def get_neighbors(self, entity: str) -> list[dict]:
        """Get all relationships for an entity."""
        neighbors = []
        
        # Outgoing edges
        for _, target, data in self.graph.out_edges(entity, data=True):
            neighbors.append({
                "direction": "outgoing",
                "predicate": data.get("predicate", "RELATED"),
                "entity": target
            })
        
        # Incoming edges
        for source, _, data in self.graph.in_edges(entity, data=True):
            neighbors.append({
                "direction": "incoming",
                "predicate": data.get("predicate", "RELATED"),
                "entity": source
            })
        
        return neighbors
    
    def find_path(self, source: str, target: str) -> list[str]:
        """Find path between two entities."""
        try:
            # Try both directions since our graph is directed
            try:
                path = nx.shortest_path(self.graph, source, target)
            except nx.NetworkXNoPath:
                path = nx.shortest_path(self.graph, target, source)
                path = list(reversed(path))
            return path
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            return []
    
    def to_text(self) -> str:
        """Convert graph to readable text."""
        lines = []
        for source, target, data in self.graph.edges(data=True):
            pred = data.get("predicate", "RELATED")
            lines.append(f"{source} {pred} {target}")
        return "\n".join(lines)
    
    def stats(self) -> dict:
        """Get graph statistics."""
        return {
            "nodes": self.graph.number_of_nodes(),
            "edges": self.graph.number_of_edges(),
            "entities_by_type": dict(Counter(self.entity_types.values()))
        }

from collections import Counter

# Build graph from sample text
kg = KnowledgeGraph()
n_entities, n_relations = kg.from_text(sample_text)

print(f"Built graph with {n_entities} entities and {n_relations} relationships")
print(f"\nGraph contents:")
print(kg.to_text())

## Part 4: Building from a Larger Corpus

In [ ]:
# Sample corpus about tech companies
corpus = [
    "Apple was founded by Steve Jobs and Steve Wozniak in 1976 in Cupertino, California.",
    "Tim Cook became CEO of Apple in 2011 after Steve Jobs resigned.",
    "Google was founded by Larry Page and Sergey Brin at Stanford University in 1998.",
    "Sundar Pichai became CEO of Google in 2015 and later CEO of Alphabet in 2019.",
    "Microsoft was founded by Bill Gates and Paul Allen in 1975 in Albuquerque.",
    "Satya Nadella became CEO of Microsoft in 2014.",
    "Amazon was founded by Jeff Bezos in 1994 in Seattle.",
    "Andy Jassy succeeded Jeff Bezos as CEO of Amazon in 2021.",
    "Tesla was founded in 2003 by Martin Eberhard and Marc Tarpenning.",
    "Elon Musk joined Tesla as chairman in 2004 and became CEO in 2008.",
    "Elon Musk also founded SpaceX in 2002 and acquired Twitter in 2022.",
    "Twitter was founded by Jack Dorsey, Noah Glass, Biz Stone, and Evan Williams in 2006.",
]

# Build comprehensive knowledge graph
tech_kg = KnowledgeGraph()

for doc in corpus:
    tech_kg.from_text(doc)

print(f"Knowledge Graph Statistics:")
print(json.dumps(tech_kg.stats(), indent=2))

print(f"\nAll relationships:")
print(tech_kg.to_text())

## Part 5: Graph-Based Question Answering

Now we can answer questions by traversing the graph.

In [ ]:
class QueryPlan(BaseModel):
    entities_to_find: list[str] = Field(description="Entities mentioned or implied in the question")
    relationship_types: list[str] = Field(description="Types of relationships to look for")
    reasoning: str = Field(description="How to answer using the graph")

def answer_with_graph(question: str, kg: KnowledgeGraph) -> dict:
    """Answer a question using the knowledge graph."""
    
    # Step 1: Parse the question to identify what to look for
    schema = QueryPlan.model_json_schema()
    schema["additionalProperties"] = False
    
    plan_response = client.responses.create(
        model=MODEL,
        input=f"""Question: {question}

Available entities in the knowledge graph: {list(kg.graph.nodes())}

Plan how to answer this question using the graph.""",
        text={
            "format": {
                "type": "json_schema",
                "name": "QueryPlan",
                "strict": True,
                "schema": schema
            }
        }
    )
    
    plan = QueryPlan.model_validate_json(plan_response.output_text)
    
    # Step 2: Gather relevant graph context
    context_parts = []
    
    for entity in plan.entities_to_find:
        # Find closest matching entity in graph
        matching = [n for n in kg.graph.nodes() if entity.lower() in n.lower()]
        
        for match in matching:
            neighbors = kg.get_neighbors(match)
            if neighbors:
                context_parts.append(f"\n{match}:")
                for n in neighbors:
                    if n["direction"] == "outgoing":
                        context_parts.append(f"  - {n['predicate']} -> {n['entity']}")
                    else:
                        context_parts.append(f"  - {n['entity']} {n['predicate']} -> [this]")
    
    graph_context = "\n".join(context_parts) if context_parts else "No relevant entities found."
    
    # Step 3: Answer using graph context
    answer_response = client.responses.create(
        model=MODEL,
        input=f"""Question: {question}

Knowledge graph context:
{graph_context}

Full graph relationships:
{kg.to_text()}

Answer the question using ONLY the information from the knowledge graph. If the information isn't in the graph, say so."""
    )
    
    return {
        "question": question,
        "plan": plan.model_dump(),
        "context": graph_context,
        "answer": answer_response.output_text
    }

# Test questions
questions = [
    "Who founded Apple?",
    "Who is the current CEO of Microsoft?",
    "What companies has Elon Musk been involved with?",
    "Where was Amazon founded?",
    "Who founded the company that Elon Musk now owns?",  # Multi-hop!
]

for q in questions:
    result = answer_with_graph(q, tech_kg)
    print(f"\nQ: {q}")
    print(f"A: {result['answer'][:300]}")

## Part 6: Multi-Hop Reasoning

Knowledge graphs excel at multi-hop questions that require traversing multiple relationships.

In [ ]:
def multi_hop_query(question: str, kg: KnowledgeGraph, max_hops: int = 3) -> dict:
    """Answer multi-hop questions by traversing the graph."""
    
    # Get all paths up to max_hops
    all_paths = []
    
    for source in kg.graph.nodes():
        for target in kg.graph.nodes():
            if source != target:
                try:
                    path = nx.shortest_path(kg.graph, source, target)
                    if len(path) <= max_hops + 1:
                        # Build path description
                        path_desc = []
                        for i in range(len(path) - 1):
                            edge_data = kg.graph.get_edge_data(path[i], path[i+1])
                            pred = edge_data.get("predicate", "RELATED") if edge_data else "RELATED"
                            path_desc.append(f"{path[i]} --[{pred}]--> {path[i+1]}")
                        all_paths.append(" | ".join(path_desc))
                except nx.NetworkXNoPath:
                    pass
    
    # Use LLM to find relevant paths and answer
    response = client.responses.create(
        model=MODEL,
        input=f"""Question: {question}

Available relationship paths in the knowledge graph:
{chr(10).join(all_paths[:50])}  # Limit for context

Find the relevant path(s) and answer the question. Show your reasoning."""
    )
    
    return {
        "question": question,
        "paths_searched": len(all_paths),
        "answer": response.output_text
    }

# Test multi-hop questions
multi_hop_questions = [
    "Who is the CEO of the company founded by Steve Jobs?",
    "What city is the headquarters of the company where Tim Cook works?",
    "Who succeeded the founder of Amazon?",
]

for q in multi_hop_questions:
    result = multi_hop_query(q, tech_kg)
    print(f"\nQ: {q}")
    print(f"Paths searched: {result['paths_searched']}")
    print(f"A: {result['answer'][:400]}")

## Part 7: When to Use Knowledge Graphs vs Vectors

| Scenario | Knowledge Graph | Vector Search | Why |
|----------|-----------------|---------------|-----|
| Multi-hop reasoning | ✓ | | Explicit relationship traversal |
| Structured domains (org charts, genealogy) | ✓ | | Natural graph structure |
| Fact verification | ✓ | | Precise relationship checking |
| Semantic similarity | | ✓ | Embeddings capture meaning |
| Unstructured content | | ✓ | No clear entities/relations |
| Large scale (millions of docs) | | ✓ | Graphs don't scale as well |
| Hybrid approach | ✓ | ✓ | Best of both worlds |

---

## Exercises

### Exercise 1: Temporal Reasoning

Extend the knowledge graph to handle temporal information ("Who was CEO of Apple in 2010?").

In [ ]:
class TemporalKnowledgeGraph(KnowledgeGraph):
    """Knowledge graph with temporal relationship support."""
    # Your implementation here
    pass

### Exercise 2: Graph + Vector Hybrid

Combine knowledge graph traversal with vector similarity for questions that need both.

In [ ]:
def hybrid_graph_vector_qa(question: str, kg: KnowledgeGraph, documents: list[str]):
    """Use graph for structured facts, vectors for context."""
    # Your implementation here
    pass

### Exercise 3: Graph Completion

Use the LLM to infer missing relationships in the graph.

In [ ]:
def infer_missing_relationships(kg: KnowledgeGraph) -> list[Relationship]:
    """Infer relationships that are likely but not explicitly stated."""
    # Your implementation here
    pass

---

## Summary

- **Entity extraction**: Use LLMs to identify named entities (people, orgs, locations)
- **Relationship extraction**: Extract (subject, predicate, object) triples
- **Graph construction**: NetworkX provides simple graph operations
- **Graph QA**: Parse question → find entities → traverse graph → generate answer
- **Multi-hop reasoning**: Follow paths through multiple relationships

Knowledge graphs are powerful for structured domains with clear entity relationships.